# Anima — preliminary subject-bucket LoRA on RTX PRO 6000 Blackwell (Colab)

Finetune **CircleStone Anima** (2B anime text-to-image DiT, NVIDIA Cosmos-Predict2-2B
backbone) with **[diffusion-pipe](https://github.com/tdrussell/diffusion-pipe)**, driven by
the `geolip_anima_trainer` package. This runs a **preliminary, rank-64 LoRA on ~1000 images**
extracted into **semantic subject buckets**, and backs checkpoints up to **HuggingFace** every
few minutes (Colab is ephemeral, so the LoRA survives a disconnect).

**What's different from a naive run (the methodology):**
- **Caption = the `caption_vlm_json` `task_1` JSON, trained VERBATIM** (this dataset teaches
  JSON-prompt conditioning), plus `caption_animetimm_json` as a hardlinked second sample.
  *Not* rendered to tags.
- **Subject buckets, not source buckets.** Images are grouped by their dominant subject;
  sparse subjects are merged by **semantic similarity** (`grp_boat`=[sailboat,boat,yacht]),
  never dropped — and distinct **human subgroups stay separate** (man/woman/player/person).
- **Anti-overtraining weighting.** `num_repeats` uses a diminishing-returns policy so sparse
  buckets are capped at ~8× (the naive equalize-to-largest policy would repeat a 5-image
  bucket ~50×/epoch → memorization).

**Runtime required:** an NVIDIA **RTX PRO 6000 Blackwell** (sm_120, 96 GB) GPU runtime — the
repo's real training target. `bf16` throughout; **no fp8, no flash-attn** (SDPA is correct on
sm_120); torch ≥ 2.7 / cu128.

> **License — read this.** Anima and *any* LoRA you train from it are **NON-COMMERCIAL**
> (CircleStone NC **+** the NVIDIA Open Model License, as a Cosmos derivative). The backup
> model card is labelled non-commercial. Keep derivatives non-commercial.

**How to run:** top to bottom. There is **one runtime restart** right after the install cell
(§2) — that is expected. Everything after the restart re-establishes its own state.

| step | section |
|---|---|
| confirm GPU | §1 |
| clone + install (→ restart) | §2 |
| verify env + `anima doctor` | §3 |
| HuggingFace auth | §4 |
| download Anima model files | §5 |
| **extract ~1000 images → semantic subject buckets** | §6 |
| **build rank-64 config + dampened weighting** | §7 |
| cache latents/text-embeds | §8 |
| train (background) + periodic HF backup | §9 |
| resume / notes | §10 |

## 1 · Confirm the GPU

You want to see **NVIDIA RTX PRO 6000 Blackwell** and ~**97 GB** of memory. If you see a T4 /
L4 / A100 instead, switch runtime (Runtime ▸ Change runtime type) — this notebook is tuned for
the Blackwell target and installs the Blackwell (cu128) torch build.

In [ ]:
!nvidia-smi

## 2 · Clone the repos and install

Clones the trainer (from GitHub, as-is) and **diffusion-pipe with submodules** (it's not
vendored on GitHub), then installs:
1. **torch + torchvision @ cu128** — required for Blackwell sm_120,
2. **diffusion-pipe's requirements** — `deepspeed`, `transformers`, `peft`, … (**no flash-attn**),
3. **light bridge deps** — `huggingface_hub`, `datasets>=2.19,<3` (the `<3` pin matters — 3.x
   changes image decoding and breaks the bridge), `Pillow`, `pyarrow`,
4. **the `[similarity]` extra** — `sentence-transformers` + `scikit-learn`, which power the
   **semantic** subject grouping in §6 (`all-MiniLM-L6-v2`, ~90 MB on first use). Without these
   the grouping silently falls back to a numpy char-trigram backend (morphological, weaker) and
   then difflib — it still never drops images, just groups them less well.

The cu128 torch and the `datasets<3` pin are re-installed **last** so neither the diffusion-pipe
requirements nor sentence-transformers can override them.

In [ ]:
import os, subprocess
WORK = "/content"
REPO = f"{WORK}/anima-trainer"
DP   = f"{REPO}/external/diffusion-pipe"
os.makedirs(WORK, exist_ok=True)

def sh(cmd):
    print("$", cmd)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit(f"command failed: {cmd}")

# anima-trainer (committed GitHub state)
if not os.path.isfile(f"{REPO}/pyproject.toml"):
    sh(f"git clone --depth 1 https://github.com/AbstractEyes/anima-trainer.git {REPO}")
else:
    print("anima-trainer already present")

# diffusion-pipe is gitignored inside anima-trainer -> clone into external/ WITH submodules
if not os.path.isfile(f"{DP}/train.py"):
    sh(f"git clone --recurse-submodules https://github.com/tdrussell/diffusion-pipe.git {DP}")
else:
    print("diffusion-pipe already present")

assert os.path.isfile(f"{DP}/train.py"), "diffusion-pipe/train.py missing — clone failed"
print("OK:", f"{DP}/train.py")

In [ ]:
# 1) Blackwell torch (cu128)
!pip -q install --index-url https://download.pytorch.org/whl/cu128 torch torchvision
# 2) diffusion-pipe reqs (deepspeed, transformers, peft; NO flash-attn)
!pip -q install -r {DP}/requirements.txt
# 3) bridge deps + the [similarity] extra (sentence-transformers pulls transformers/sklearn;
#    it may pin huggingface_hub<1 — harmless for the bridge). Keep on ONE line each (! magic).
!pip -q install "huggingface_hub>=0.23" "Pillow>=10" "pyarrow>=14" "sentence-transformers>=2.7,<4" "scikit-learn"
# 4) re-pin datasets<3 and cu128 torch LAST so nothing above overrode them
!pip -q install "datasets>=2.19,<3"
!pip -q install --index-url https://download.pytorch.org/whl/cu128 torch torchvision
print("\n✅ installs done.")
print("⚠️  RESTART THE RUNTIME now so the new torch loads: run the next cell, or Runtime ▸ Restart session.")

**Restart the runtime.** Run the cell below (it force-restarts the kernel; Colab reconnects
automatically). After it restarts, **continue at §3** — do not re-run §2.

In [ ]:
# One-shot restart so the freshly-installed cu128 torch is the one that loads.
import os
os.kill(os.getpid(), 9)

## 3 · (after restart) Verify the environment

Re-establishes paths, puts the package on `sys.path` (we run it **from source** — no
`pip install` needed, which also sidesteps the package's Python ≥ 3.12 gate), then checks torch
sees Blackwell + cu128 + bf16, and runs `anima doctor`.

In [ ]:
import os, sys, subprocess
WORK = "/content"
REPO = f"{WORK}/anima-trainer"
DP   = f"{REPO}/external/diffusion-pipe"
sys.path.insert(0, REPO)                                   # import geolip_anima_trainer from source
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

import torch
ok_cuda = torch.cuda.is_available()
name = torch.cuda.get_device_name(0) if ok_cuda else "NONE"
cap  = torch.cuda.get_device_capability(0) if ok_cuda else (0, 0)
sm   = cap[0] * 10 + cap[1]
print(f"torch {torch.__version__} | cuda build {torch.version.cuda} | device {name} | sm_{sm} "
      f"| bf16 {torch.cuda.is_bf16_supported() if ok_cuda else False}")

assert ok_cuda, "No CUDA device — set a GPU runtime."
cb = tuple(int(x) for x in (torch.version.cuda or '0.0').split('.')[:2])
assert cb >= (12, 8), (f"torch cuda {torch.version.cuda} < 12.8 — re-run the §2 install cell, "
                       f"then the restart cell, then this one.")
if sm < 120:
    print(f"⚠️  Detected sm_{sm}, not Blackwell sm_120. It may run, but this notebook is tuned for "
          f"the RTX PRO 6000 Blackwell.")

In [ ]:
import geolip_anima_trainer as anima
print("geolip_anima_trainer", anima.__version__, "->", anima.__file__)

def anima_cli(*args):
    """Run the `anima` CLI via module path (no console-script / PATH / py-version dependency)."""
    cmd = [sys.executable, "-m", "geolip_anima_trainer.cli", *map(str, args)]
    print("›", " ".join(cmd))
    return subprocess.run(cmd, cwd=REPO).returncode

# Environment diagnostics: torch cu128 / sm_120 / bf16 / flash-attn absent / deepspeed / diffusion-pipe.
print(anima.doctor(repo_root=REPO).render())

## 4 · HuggingFace authentication

Add a **WRITE** token to Colab **Secrets** (🔑 in the left sidebar) named **`HF_TOKEN`**, then
run this. The token is used to read the dataset/model (public, but auth avoids rate limits) and
to **push checkpoint backups**. The backup repo is auto-named under your account.

In [ ]:
import os
def get_hf_token():
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t:
            return t
    except Exception:
        pass
    return os.environ.get("HF_TOKEN")

HF_TOKEN = get_hf_token()
assert HF_TOKEN, "No HF_TOKEN. Add it in Colab Secrets (🔑) with WRITE scope, then re-run."

from huggingface_hub import login, whoami
login(token=HF_TOKEN, add_to_git_credential=True)
HF_USER = whoami(token=HF_TOKEN)["name"]
BACKUP_REPO = f"{HF_USER}/anima-prelim-1k-r64"      # <- edit if you want a different name
print("logged in as", HF_USER, "| backups ->", BACKUP_REPO)

## 5 · Download the Anima model files (~5.6 GB)

Fetches the three ComfyUI-format files into `models/anima/` via `huggingface_hub` and returns
their resolved paths (used to build the `[model]` block). Base checkpoint: **anima-base-v1.0**.

In [ ]:
MODELS_DIR = f"{REPO}/models/anima"
paths = anima.download_models(MODELS_DIR, base="base-v1.0")   # -> ModelPaths
print("transformer:", paths.transformer_path)
print("vae        :", paths.vae_path)
print("llm        :", paths.llm_path)

## 6 · Extract ~1000 images → semantic subject buckets

`anima subjects` does a **columnar** pyarrow read of the dataset shard(s) and writes the
`caption_vlm_json` **verbatim** into each `.txt` sidecar (+ `caption_animetimm_json` as a
hardlinked second sample when it's a real JSON), bucketing images by their **dominant subject**.
Settings resolved for `qwen_90k` (see `CLAUDE.md`):

- **caption = `caption_vlm_json` trained verbatim** (task_1 JSON), `caption_animetimm_json` as a
  2nd sample — both are well-populated for this config,
- **`require_age_pass=False`** — `age_classifier_pass` is unpopulated here, so the default gate
  would drop *every* row,
- **audit gate ON** (`audit == "approved"`),
- **`limit=1000`** → stops after the first shard (~1.3 GB); never scans the whole ~100 GB config.

**Grouping (the new bit):** sparse subjects are merged by **semantic similarity** into weighted
`grp_*` buckets instead of being dropped, while large buckets and **distinct human subgroups**
(man/woman/player/person/…) are protected and kept separate. Genuine singletons pool into a
weighted `misc_*` catch-all — nothing is omitted. Real semantic grouping needs the `[similarity]`
extra from §2; otherwise it falls back to char-trigram/difflib (you'll see the tier logged).

> On the very diverse `sdxl_qwen_phase0` slice most dominant subjects are unique, so `misc_other`
> can be large — that's fine, it's weighted ~1× (kept, not overtrained). For a more focused
> source it shrinks. `--build-toml` / the §7 build then balances everything.

In [ ]:
SUBJECTS_ROOT = f"{REPO}/datasets/anima_subjects"

rep = anima.export_subject_buckets(anima.SubjectBucketConfig(
    repo="AbstractPhil/diffusion-pretrain-set-ft1",
    config="qwen_90k",
    out_root=SUBJECTS_ROOT,
    limit=1000,                        # global cap -> stops after the first shard
    require_audit_approved=True,       # audit == "approved" (populated)
    require_age_pass=False,            # age_classifier_pass unpopulated -> MUST be off
    # --- semantic grouping (needs the [similarity] extra; else trigram/difflib fallback) ---
    use_semantic=True,
    similarity_model="sentence-transformers/all-MiniLM-L6-v2",  # or nomic-...-v1 for 0 download
    min_bucket_size=10,                # >= this many images -> a protected "large" bucket
    min_final_group_size=8,            # a semantic group below this -> weighted misc_*
    keep_small=True,                   # weight, don't drop: leftovers -> misc_*
))
print(f"accepted={rep['accepted_images']}  dropped={rep['dropped_images']}  "
      f"raw_subjects={rep['raw_subjects']} -> buckets={rep['n_final_buckets']}")
print("caption stats:", rep["caption_stats"])
print("\nbuckets (samples each):")
for k, v in list(rep["final_buckets"].items())[:24]:
    print(f"  {k:<22}{v}")

*(Optional)* probe the source first — caption fill rates and the audit/age gate
distributions on a small sample:

In [ ]:
# info = anima.inspect_source("AbstractPhil/diffusion-pretrain-set-ft1", "qwen_90k", n=64, verbose=True)

## 7 · Build the rank-64 config + dampened weighting

Discovers the subject-bucket folders from §6 and balances `num_repeats` with the
**diminishing-returns** policy (`BALANCE_ALPHA=0.5`, sqrt): big buckets stay ~1–2×, sparse/grouped
buckets are lifted but **capped at `MAX_REPEATS=8`** (and at `CAP_MULT×top` effective) so they
contribute without overtraining. It then writes **both** `anima_lora.toml` and `anima_dataset.toml`
through the package's single validating writer, which enforces the Anima invariants:
**`llm_adapter_lr = 0`** (adapter frozen) and **`shuffle_caption = false`** (tag-order sensitive).

> `BALANCE_ALPHA=0` reproduces the legacy equalize-to-largest behavior (which overtrains sparse
> buckets); `1.0` disables balancing. `0.5` is the recommended default. Edit the tunables to taste.

In [ ]:
from geolip_anima_trainer.config import discover_directories

# ---- tunables (preliminary LoRA) -------------------------------------------------
RANK                = 64           # multi-concept
LR                  = 2e-5         # Anima likes LOW lr
MICRO_BATCH         = 4            # 96 GB can go higher (8); 4 is safe
EPOCHS              = 20           # preliminary; bump once you trust the run
SAVE_EVERY_N_EPOCHS = 5            # a usable LoRA every 5 epochs
WARMUP_STEPS        = 50
RES                 = 1024
# diminishing-returns weighting (anti-overtraining):
BALANCE_ALPHA       = 0.5          # 0=equalize(legacy/overtrains), 1=no balance, 0.5=sqrt
MAX_REPEATS         = 8            # per-image exposure ceiling (was 50 under equalization)
CAP_MULT            = 1.25         # cap a bucket's effective samples at CAP_MULT * top
TARGET_EFFECTIVE    = None         # None -> top = the largest bucket's image count
OUTPUT_DIR          = "/content/runs/anima_prelim"
CONFIGS_DIR         = f"{REPO}/configs/colab_prelim"
# ----------------------------------------------------------------------------------

dirs = discover_directories(SUBJECTS_ROOT)
assert dirs, f"No subject-bucket folders under {SUBJECTS_ROOT}; did §6 run?"

ds = anima.DatasetConfig(resolutions=[RES], directories=dirs,
                         balance_alpha=BALANCE_ALPHA, cap_mult=CAP_MULT,
                         max_repeats=MAX_REPEATS, target_effective=TARGET_EFFECTIVE)
anima.rebalance(ds)                                   # fills num_repeats via dampened_repeats

model = anima.ModelConfig(transformer_path=paths.transformer_path,
                          vae_path=paths.vae_path, llm_path=paths.llm_path,
                          llm_adapter_lr=0.0)         # FROZEN — non-negotiable for Anima
run = anima.RunConfig(output_dir=OUTPUT_DIR, epochs=EPOCHS,
                      micro_batch_size_per_gpu=MICRO_BATCH,
                      save_every_n_epochs=SAVE_EVERY_N_EPOCHS, warmup_steps=WARMUP_STEPS)
cfg = anima.TrainConfig(run=run, model=model,
                        adapter=anima.AdapterConfig(rank=RANK),
                        optimizer=anima.OptimizerConfig(lr=LR), dataset=ds)

LORA_TOML, DATASET_TOML = anima.render_train_toml(cfg, out_dir=CONFIGS_DIR)  # validates + writes both
LORA_TOML, DATASET_TOML = str(LORA_TOML), str(DATASET_TOML)
print("wrote:", LORA_TOML)
print("wrote:", DATASET_TOML)
print("\nbalance (num_repeats capped at %d, no 50x):" % MAX_REPEATS)
for d in sorted(ds.directories, key=lambda d: -(d.image_count or 0)):
    n = d.image_count or 0
    print(f"  {(d.name or ''):<24} imgs={n:<5} x{d.num_repeats:<2} = {n*d.num_repeats} effective")
print("total effective samples/epoch:", sum((d.image_count or 0)*d.num_repeats for d in ds.directories))

## 8 · Cache latents + text embeddings

`deepspeed … train.py --deepspeed --config … --cache_only`. VAE latents and Qwen-3 text
embeddings are computed once and cached next to the images, then reused by training. The first
line previews the exact command; the second runs it.

In [ ]:
anima.cache(LORA_TOML, repo_root=REPO, num_gpus=1, dry_run=True)    # preview the command

In [ ]:
rc = anima.cache(LORA_TOML, repo_root=REPO, num_gpus=1)             # build the cache (blocking)
print("cache exit code:", rc)

## 9 · Train (background) + periodic HuggingFace backup

Training is launched as a **background process** so a second cell can **back up checkpoints to
HuggingFace every few minutes** while it runs — that's what makes the LoRA survive a Colab
disconnect. Saved LoRAs land in `OUTPUT_DIR/<timestamp>/epochN/` (ComfyUI-format safetensors +
PEFT config); deepspeed's bulky `global_step*` training-state dirs are skipped from the backup
by default (flip `INCLUDE_RESUME_STATE` if you want full `--resume_from_checkpoint` capability).

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi, create_repo

INCLUDE_RESUME_STATE = False   # True = also back up deepspeed global_step* (large; needed to resume across runtimes)

_ROOT_CARD = '''---
license: other
license_name: anima-non-commercial
license_link: https://huggingface.co/circlestone-labs/Anima
base_model: circlestone-labs/Anima
tags:
- lora
- diffusion-pipe
- anima
- cosmos-predict2
- text-to-image
- non-commercial
pipeline_tag: text-to-image
---

# Anima preliminary LoRA (rank 64, semantic subject buckets)

Preliminary LoRA finetune of **CircleStone Anima** (2B DiT, NVIDIA Cosmos-Predict2-2B backbone),
trained with [diffusion-pipe](https://github.com/tdrussell/diffusion-pipe) via the
`geolip_anima_trainer` package on a ~1000-image slice of
`AbstractPhil/diffusion-pretrain-set-ft1` (`qwen_90k`).

**Methodology:** images are organized into **semantic subject buckets**; the `task_1` JSON
caption (`caption_vlm_json`, + `caption_animetimm_json` as a 2nd sample) is trained **verbatim**
(this teaches JSON-prompt conditioning). Sparse subjects are grouped by semantic similarity and
weighted with a diminishing-returns `num_repeats` policy (capped ~8×) so they don't overtrain;
distinct human subgroups are kept separate. `llm_adapter_lr = 0` (adapter frozen).

> **NON-COMMERCIAL.** Derivative of Anima — inherits CircleStone's non-commercial terms **and**
> the NVIDIA Open Model License (Cosmos derivative). Do not use commercially.

Each `runs/<timestamp>/epochN/` is a ComfyUI-format LoRA — drop into `ComfyUI/models/loras/`.
'''

def _latest_run_dir(output_dir):
    p = Path(output_dir)
    if not p.is_dir():
        return None
    runs = [d for d in p.iterdir() if d.is_dir() and (any(d.glob("epoch*")) or any(d.glob("global_step*")))]
    return max(runs, key=lambda d: d.stat().st_mtime) if runs else None

def anima_hf_backup(repo_id, output_dir, *, token, include_resume_state=INCLUDE_RESUME_STATE):
    """Upload the latest run dir (epoch* LoRAs + tensorboard + config) to a HF model repo.
    Idempotent: upload_folder hashes files, so unchanged ones are skipped on repeat calls."""
    run = _latest_run_dir(output_dir)
    if run is None:
        print("  [backup] no checkpoints yet under", output_dir)
        return None
    api = HfApi(token=token)
    create_repo(repo_id, token=token, repo_type="model", private=True, exist_ok=True)
    api.upload_file(path_or_fileobj=_ROOT_CARD.encode(), path_in_repo="README.md",
                    repo_id=repo_id, repo_type="model", commit_message="model card (non-commercial)")
    ignore = None if include_resume_state else ["global_step*/*", "global_step*/**"]
    api.upload_folder(folder_path=str(run), repo_id=repo_id, repo_type="model",
                      path_in_repo=f"runs/{run.name}", ignore_patterns=ignore,
                      commit_message=f"backup :: {run.name}")
    url = f"https://huggingface.co/{repo_id}/tree/main/runs/{run.name}"
    print("  [backup] ->", url)
    return url

print("backup helper ready. INCLUDE_RESUME_STATE =", INCLUDE_RESUME_STATE)

**Start training** (non-blocking). The exact command is printed first; output is tee'd to
`OUTPUT_DIR/train.log`.

In [ ]:
from geolip_anima_trainer import launch as _launch

plan = _launch.build_plan(config_toml=LORA_TOML, repo_root=REPO, num_gpus=1)
print(plan.batch_summary())
print(plan.pretty())

os.makedirs(OUTPUT_DIR, exist_ok=True)
TRAIN_LOG = f"{OUTPUT_DIR}/train.log"
_env = {**os.environ, **plan.env_prefix()}
_logf = open(TRAIN_LOG, "w")
train_proc = subprocess.Popen([str(a) for a in plan.argv()], cwd=str(plan.train_py.parent),
                              env=_env, stdout=_logf, stderr=subprocess.STDOUT, text=True)
print("\ntraining PID", train_proc.pid, "| log:", TRAIN_LOG)

**Monitor + back up.** This cell tails the log and pushes a backup every `BACKUP_EVERY_MIN`
minutes until training exits, then does a final backup. Safe to re-run if it gets interrupted —
training keeps running in the background regardless.

In [ ]:
import time

BACKUP_EVERY_MIN = 15

try:
    while train_proc.poll() is None:
        time.sleep(BACKUP_EVERY_MIN * 60)
        try:
            print(open(TRAIN_LOG).read()[-1500:])
        except FileNotFoundError:
            pass
        anima_hf_backup(BACKUP_REPO, OUTPUT_DIR, token=HF_TOKEN)
except KeyboardInterrupt:
    print("monitor stopped (training continues in background; re-run this cell to resume monitoring)")

print("\n=== training exited with code", train_proc.returncode, "===")
print(open(TRAIN_LOG).read()[-3000:])
anima_hf_backup(BACKUP_REPO, OUTPUT_DIR, token=HF_TOKEN)   # final backup

## 10 · Resume, re-run, and notes

**Manual backup anytime** (e.g. you stopped training early):
```python
anima_hf_backup(BACKUP_REPO, OUTPUT_DIR, token=HF_TOKEN)
```

**Resume training** after an interruption (same runtime, checkpoints still on disk) — re-run the
*start training* cell with `--resume_from_checkpoint`:
```python
plan = _launch.build_plan(config_toml=LORA_TOML, repo_root=REPO, num_gpus=1,
                          resume_from_checkpoint=True)   # or "<timestamp>" for a specific run
```
To resume on a **fresh** runtime you need the deepspeed `global_step*` state — set
`INCLUDE_RESUME_STATE = True` in §9 so it gets backed up, then download it back into `OUTPUT_DIR`.

**Get the LoRA:** it's at `https://huggingface.co/<you>/anima-prelim-1k-r64` under
`runs/<timestamp>/epochN/` — drop the `epochN` safetensors into `ComfyUI/models/loras/`.

**Scaling up later:**
- Raise `EPOCHS`, and raise `--limit` / `SubjectBucketConfig(limit=...)` (or drop it to extract
  more shards). The full config is ~100 GB across 56 shards, so a big extraction belongs on a
  roomy disk, not Colab scratch.
- Keep `BALANCE_ALPHA=0.5` and `MAX_REPEATS=8` so the long tail of sparse buckets stays weighted
  down; lower `min_final_group_size` / loosen `sim_threshold` to fold more singletons into
  semantic groups (and shrink `misc_other`).
- Zero-download similarity backend: pass `similarity_model="nomic-ai/nomic-embed-text-v1"` if
  it's already in the HF cache (otherwise MiniLM is a one-time ~90 MB fetch).
- Keep `llm_adapter_lr = 0` unless you're deliberately A/B-testing the two-phase adapter protocol
  (ceiling 5e-6 — `validate()` hard-errors above it).

**Even more durable than backups:** point `OUTPUT_DIR` at a mounted Google Drive folder so
checkpoints persist on disk independently of the runtime:
```python
from google.colab import drive; drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/anima_runs/prelim"   # set this in §7 before building the config
```

> **License reminder:** the LoRA is a non-commercial Anima derivative (CircleStone NC + NVIDIA
> Open Model License). The backup model card says so.